In [7]:
import numpy as np
import pandas as pd
import networkx as nx
from networkx.algorithms import community

from sklearn.metrics.pairwise import cosine_similarity
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

In [39]:
pip install "numpy<2.0"

In [2]:
from node2vec import Node2Vec

c:\Users\HOME\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
# 1. Load data
df = pd.read_csv('../../Dataset/edge_260429.csv')
df.head()

,Source,Target,Count
0,1분마뫄,강지,1.00
1,1분마뫄,러너,11.00
2,1분마뫄,백곰파,0.25
3,1분마뫄,빅헤드,3.00
4,1분마뫄,사모장,0.25


In [5]:
# 2. Weighted Graph
# 방향성이 없는 그래프(Graph)로 가정합니다. 방향이 중요하다면 nx.DiGraph()를 사용합니다.
G = nx.from_pandas_edgelist(
    df, 
    source='Source', 
    target='Target', 
    edge_attr='Count', # weight
    create_using=nx.DiGraph()
)

print(f"Total nodes: {G.number_of_nodes()}\nTotal edges: {G.number_of_edges()}\n")


Total nodes: 186
Total edges: 3390



# Community Detection (Directed graph)

- Louvain method
- hyper-para: `resolution`

In [18]:
resolution = 1.0
communities = nx.community.louvain_communities(G, weight='Count', resolution=resolution, seed=42)

for i, comm in enumerate(communities):
    # comm은 set 형태이므로 보기 편하게 list로 변환하여 출력합니다.
    print(f"{i+1}-th community: (Total {len(comm)} nodes): \n{list(comm)}\n")

1-th community: (Total 28 nodes): 
['플레임[이호종]', '순당무', '도티 TV', '풍월량', '랄로', 'Bang', '인간젤리', '쉐리', '러너', '일루전 ILLUSION', '군림보', '텔론', '따효니DDaHyoNi', '밍모', 'T1 Faker', '피닉스박', '괴물쥐 유튜브', '1분마뫄', '명예훈장', '테스터훈 TesterHoon', '세찬 SECHAN', '동수칸TV', '포피셜', '우주하마', 'PAKA', '아이치', 'Official dopa', '얍얍']

2-th community: (Total 8 nodes): 
['아론 페이지', '콩콩이다', '삼식', '후추', '다주', '서넹', '루태TV', '양띵 유튜브']

3-th community: (Total 62 nodes): 
['채현찌', '노돌리', '윤이샘', '시라유키 히나 SHIRAYUKI HINA', '푸린', '임나은', '행돌', '설백', '나나양', '삐부찡', '아야츠노 유니 AYATSUNO YUNI', '유즈하 리코 YUZUHA RIKO', '치킨쿤', '금휘', '연비니', '감블러', '새옴', '금사향', '댕균', '코렛트YouTube', '꽃핀', 'Youtube고차비', '이초홍', '사키하네 후야 SAKIHANE HUYA', '램램', '코시', '아오쿠모 린 AOKUMO RIN', '나부', 'MadLife TV', '아구 TV', '악동 김블루', '아카네 리제 AKANE LIZE', '망개', '원정상', '아라하시 타비 ARAHASHI TABI', '네네코 마시로 NENEKO MASHIRO', '강지', '아구이뽀 A924', 'Wolf', '이춘향', '탬탬버린', '쫀득TV', '빅헤드', '윤가놈', '유콘', '백곰파', '김뿡', '강퀴TV', 'YouTube지누', '코뚱잉', '류제홍 RYUJEHONG', '악녀', '실프', '하나코 나나 HANAKO NANA', '뜨뜨뜨뜨',

***

# Link Prediction (Un-directed graph)

## Link Concepts

1. **Positive** : Linked nodes' pair (1)
2. **Negative** : Non-edged nodes' pair (0)

## Link prediction - (1) Non-edge pair: Scores?

> Negative : Non-edged nodes' pair 만 사용

- Node2Vec
  - Cosine Similarity
  - Dot Product
  - $L_2$ norm

In [ ]:
# No Directed Graph
G_un = G.to_undirected()

node2vec = Node2Vec(
    G_un,
    dimensions=64,   # 임베딩 벡터 차원
    walk_length=20,  # 랜덤 워크 길이
    num_walks=100,   # 노드당 워크 수
    p=1,             # Return 파라미터 (BFS - DFS 조절)
    q=1,             # In-out 파라미터
    weight_key='Count',
    workers=1
)

model = node2vec.fit(window=10, min_count=1)

Generating walks (CPU: 1): 100%|██████████| 100/100 [00:02<00:00, 38.29it/s]


In [31]:
# 존재하지 않는 엣지 쌍 추출
non_edges = list(nx.non_edges(G_un))

results = []
for u, v in non_edges:
    vec_u = model.wv[str(u)]  # 노드 id가 str이 아니면 str() 필요
    vec_v = model.wv[str(v)]
    
    cos = cosine_similarity([vec_u], [vec_v])[0][0]  # cosine_similarity는 2D 배열을 반환하므로 [0][0]으로 스칼라 값 추출
    dot  = float(np.dot(vec_u, vec_v))
    l2   = np.linalg.norm(vec_u - vec_v)
    
    results.append({'node1': u, 'node2': v, 'cosine_sim': cos, 'dot': dot, 'l2': l2})

link_pred = pd.DataFrame(results)

### Cosine Similarity $\in [-1, 1]$

- just direction check
- normalization $\to$ node importance 무시

In [32]:
print(link_pred.sort_values('cosine_sim', ascending=False).head(10)[['node1','node2','cosine_sim']])


       node1  node2  cosine_sim
12772     우융     조밈    0.947570
12804     우융    파크모    0.938698
6624     이비온  퀸톨 TV    0.936305
7972      각별  케인 TV    0.924437
13517  잠뜰 TV  케인 TV    0.906162
13554  잠뜰 TV    파크모    0.904368
14952  케인 TV    파크모    0.903492
3063      코마  케인 TV    0.901928
14735     조밈    파크모    0.888706
6690     이비온  코아 TV    0.887708


- small node도 집계됨 (just direction)

### Dot Product $\in \mathbb{R}$

- node importance 고려

In [35]:
print(link_pred.sort_values('dot', ascending=False).head(10)[['node1','node2','dot']])


                      node1                 node2        dot
5306               감스트GAMST                   아오니  17.991100
13286                 두치와뿌꾸                   아오니  17.582584
5231               감스트GAMST                    혜안  17.432951
9903                     혜안                 두치와뿌꾸  17.017509
13009  비올레타 모네 Violeta Mone                 두치와뿌꾸   9.991445
5270               감스트GAMST  비올레타 모네 Violeta Mone   9.973799
2947                     코마              감스트GAMST   9.795331
10602           류시호 RyuSiho                 두치와뿌꾸   9.716150
3025                     코마                 두치와뿌꾸   9.703435
5238               감스트GAMST           류시호 RyuSiho   9.677392


- hub node 집계됨: vector size 반영

In [36]:
degrees = pd.DataFrame(G_un.degree(weight='Count'), columns=['node', 'weighted_degree'])
print(degrees.sort_values('weighted_degree', ascending=False).head(10))

       node  weighted_degree
73       다주           887.50
36   양띵 유튜브           857.00
65       후추           801.50
19       삼식           774.00
64     콩콩이다           770.75
6        서넹           749.00
76     루태TV           649.25
127  악어 유튜브           566.75
52       너불           556.25
86       핑맨           534.75


- 주요 nodes' degree?

In [37]:
nodes_of_interest = ['감스트GAMST', '두치와뿌꾸', '아오니', '다주', '양띵 유튜브', '후추']
for n in nodes_of_interest:
    print(f"{n}: {G_un.degree(n, weight='weight')}")

감스트GAMST: 1
두치와뿌꾸: 1
아오니: 1
다주: 62
양띵 유튜브: 70
후추: 69


- Analysis
- 감스트, 두치, 아오니: 서로 많이 했을 것으로 추정. 좁은 관계
- 다주, 양띵, 후추: 다양한 유튜버와 많이 했을 것으로 추정, 넓은 관계.
- 따라서 dot product 결과와 (link pred) degree 결과 (counts)가 다르게 판단됨.

### $L_2$ norm $\in \mathbb{R}_+$

In [30]:
print(link_pred.sort_values('l2', ascending=True).head(10)[['node1','node2','l2']])


          node1  node2        l2
12772        우융     조밈  1.016228
6624        이비온  퀸톨 TV  1.053054
12804        우융    파크모  1.090763
7972         각별  케인 TV  1.181334
90        멋사유튜브  수닝유튜브  1.263383
14952     케인 TV    파크모  1.296332
13517     잠뜰 TV  케인 TV  1.315019
13554     잠뜰 TV    파크모  1.317305
6690        이비온  코아 TV  1.317750
10848  마젯 Mazet    하지유  1.326059


- cosine similiarity와 비슷한 결과
- distance는 그다지 좋은 지표가 아닌듯 함.

***

## Link prediction - (2) Edge pair: Classification Problem

1. **Positive** : Linked nodes' pair (1)
2. **Negative** : Non-edged nodes' pair (0)

모두 사용

- Count 최소단위: 0.25 = 0.5 $\gamma$
- Classification Models
  - Count > 0
  - Count == 0
  - training 의미 X. Additional features?